# DentalCare Account Manager - Colab Version

## Installation Options

**Option A:** Use Google Drive (Recommended - easiest)
- Upload `colab_version.zip` to your Google Drive
- Get share link and paste below

**Option B:** Use Public URL (Direct download)
- Upload ZIP to any file hosting (Mega, Dropbox, etc.)
- Get direct download link and paste below

**Option C:** GitHub (Original)
- Use the GitHub repo URL

In [ ]:
# ============================================
# CONFIGURATION - CHOOSE ONE METHOD BELOW
# ============================================

# METHOD A: Google Drive Share Link
# 1. Upload colab_version.zip to Google Drive
# 2. Right-click > Share > Anyone with link
# 3. Copy the share link and paste below
DRIVE_SHARE_LINK = ""  # e.g., "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"

# METHOD B: Direct Public URL (Mega, Dropbox, etc.)
# Get the direct download URL
PUBLIC_ZIP_URL = ""  # e.g., "https://example.com/colab_version.zip"

# METHOD C: GitHub Repository
GITHUB_REPO_URL = "https://github.com/USERNAME/REPO"  # Leave empty if using other methods

# ============================================
# RUNTIME SETTINGS
# ============================================

# Number of parallel browsers (recommended: 3-5 for free Colab)
PARALLEL_COUNT = 3  # @param {type:"integer"}

# Number of accounts to process (0 = unlimited)
BATCH_COUNT = 0  # @param {type:"integer"}

# Delay between batches (seconds)
DELAY = 2  # @param {type:"integer"}

# Your MFA phone service URL
MFA_PHONE_URL = "http://localhost:6729/get-number"  # @param {type:"string"}

print(f"Configuration loaded:")
print(f"  Parallel Count: {PARALLEL_COUNT}")
print(f"  Batch Count: {'Unlimited' if BATCH_COUNT == 0 else BATCH_COUNT}")
print(f"  Delay: {DELAY}s")
print(f"  Phone API: {MFA_PHONE_URL}")

## Step 1: Install Dependencies

In [ ]:
!pip install playwright requests > /dev/null 2>&1
!playwright install chromium > /dev/null 2>&1
print("Dependencies installed!")

## Step 2: Download/Extract Application

In [ ]:
import os
import subprocess

# Determine which method to use
download_method = None
download_source = None

if DRIVE_SHARE_LINK:
    download_method = 'drive'
    download_source = DRIVE_SHARE_LINK
    print(f"Using Google Drive: {DRIVE_SHARE_LINK[:50]}...")
elif PUBLIC_ZIP_URL:
    download_method = 'url'
    download_source = PUBLIC_ZIP_URL
    print(f"Using Public URL: {PUBLIC_ZIP_URL[:50]}...")
elif GITHUB_REPO_URL:
    download_method = 'github'
    download_source = GITHUB_REPO_URL
    print(f"Using GitHub: {GITHUB_REPO_URL}")
else:
    print("ERROR: No download source specified!")
    print("Please set DRIVE_SHARE_LINK, PUBLIC_ZIP_URL, or GITHUB_REPO_URL")
    raise ValueError("No download source")

# Cleanup any existing directory
app_dir = '/content/colab_version'
if os.path.exists(app_dir):
    !rm -rf {app_dir}

# Download based on method
if download_method == 'drive':
    # Extract file ID from Google Drive share link
    file_id = DRIVE_SHARE_LINK.split('/d/')[1].split('/')[0] if '/d/' in DRIVE_SHARE_LINK else None
    if file_id:
        print(f"Downloading from Google Drive (ID: {file_id})")
        !gdown {file_id} -O /content/colab_version.zip
    else:
        print("Invalid Google Drive link format")

elif download_method == 'url':
    print(f"Downloading from: {PUBLIC_ZIP_URL}")
    !wget -q "{PUBLIC_ZIP_URL}" -O /content/colab_version.zip

elif download_method == 'github':
    repo_name = GITHUB_REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
    print(f"Cloning GitHub repository: {repo_name}")
    !git clone {GITHUB_REPO_URL} /content/{repo_name} 2>&1
    app_dir = f'/content/{repo_name}'

# Extract ZIP if downloaded
if download_method != 'github' and os.path.exists('/content/colab_version.zip'):
    print("Extracting ZIP...")
    !unzip -q /content/colab_version.zip -d /content/
    # Find the extracted directory
    for item in os.listdir('/content/'):
        if os.path.isdir(f'/content/{item}') and item != 'colab_version.zip':
            if 'colab' in item.lower() or 'dental' in item.lower():
                app_dir = f'/content/{item}'
                break
    else:
        # Use first extracted directory
        extracted = [d for d in os.listdir('/content/') if os.path.isdir(f'/content/{d}') and d != 'colab_version.zip']
        if extracted:
            app_dir = f'/content/{extracted[0]}'

print(f"\nApplication directory: {app_dir}")
print(f"Files: {os.listdir(app_dir)[:10]}...")

## Step 3: Verify Installation

In [ ]:
import sys
sys.path.insert(0, app_dir)

# Test imports
try:
    import config
    import models
    from services import api_registration_service, browser_flow_service, mfa_service, phone_service, worker_service
    print("All modules imported successfully!")
except Exception as e:
    print(f"Import error: {e}")

## Step 4: Run the Account Manager

In [ ]:
import asyncio
import logging
from datetime import datetime

# Change to the app directory
os.chdir(app_dir)
print(f"Working directory: {os.getcwd()}")

# Import and run the application
from run import ColabRunner

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Create and run the runner
runner = ColabRunner(
    parallel_count=PARALLEL_COUNT,
    batch_count=BATCH_COUNT,
    delay=DELAY,
    phone_api_url=MFA_PHONE_URL
)

print("\n" + "="*60)
print("STARTING ACCOUNT MANAGER")
print("="*60)
print("\nNOTE: Press the STOP button or Runtime > Interrupt execution to stop\n")

# Run the application
loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)
try:
    loop.run_until_complete(runner.run())
finally:
    loop.close()

## Step 5: Download Results

In [ ]:
import glob

# Find the latest results file
results_dir = f'{app_dir}/data/results'
result_files = glob.glob(f'{results_dir}/accounts_*.json')

if result_files:
    latest_file = max(result_files, key=os.path.getmtime)
    print(f"Latest results: {latest_file}")
    print(f"\nAccount count: {len(open(latest_file).readlines())}")
    print(f"\nTo download results, run:\n")
    print(f"from google.colab import files")
    print(f"files.download('{latest_file}')")
else:
    print("No results found yet.")

---

## Quick Reference

| Method | Pros | Cons |
|--------|------|------|
| Google Drive | Fast, Easy | Need to upload first |
| Public URL | Works anywhere | Need stable hosting |
| GitHub | Version control | Need public repo |

### Resource Limits

| Plan | Max Browsers | Est. Accounts/12hr |
|------|-------------|-------------------|
| Free | 3-5 | 2,000-4,000 |
| Pro | 10-15 | 5,000-8,000 |